# 🗺️ Módulo 09 - Notebook 03: Spatial joins y buffers comerciales

## 🎯 Análisis de cobertura y áreas de influencia

**Libro:** Saliendo de lo Pandito  
**Módulo:** 09 - Analítica Geoespacial GeoPandas  
**Duración estimada:** 75 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** buffers (áreas de influencia) alrededor de puntos  
✅ **Realizar** spatial joins (uniones espaciales)  
✅ **Analizar** coberturas territoriales  
✅ **Identificar** superposiciones entre zonas  
✅ **Optimizar** ubicaciones comerciales

---

## 📋 Pre-requisitos

* ✅ Notebooks 09_01 y 09_02 completados
* ✅ Conocimiento de GeoDataFrames y CRS
* ✅ Familiaridad con operaciones espaciales

---

## 📚 Contenido

1. Teoría de Buffers
2. Creación de Áreas de Influencia
3. Spatial Joins (sjoin)
4. Análisis de Cobertura
5. Detección de Superposiciones
6. Caso Integrador: Optimización de Ubicaciones

---

## 💡 Por qué importa

**Análisis espacial transforma decisiones de negocio:**

* 🏪 **Retail:** ¿Dónde abrir la próxima sucursal sin canibalizar ventas?
* 🚚 **Logística:** ¿Qué sucursal cubre cada cliente?
* 🏦 **Banca:** ¿Dónde hay zonas sin cobertura?
* 📏 **Marketing:** Segmentación territorial por alcance

**La ubicación es el dato más valioso del negocio**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Extraer ubicaciones únicas de sucursales
    df_sucursales = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    # Agregar ventas totales por sucursal
    ventas_totales = df_ventas.groupby('sucursal_id')['ventas'].sum().reset_index()
    ventas_totales.columns = ['sucursal_id', 'ventas_totales']
    df_sucursales = df_sucursales.merge(ventas_totales, on='sucursal_id')
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros ventas: {len(df_ventas):,}")
    print(f"   🏪 Sucursales: {len(df_sucursales)}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n🎯 Datos espaciales:")
    print(f"   • Coordenadas: {len(df_sucursales)} sucursales")
    print(f"   • Listo para crear buffers (áreas de influencia)")
    print(f"   • Listo para spatial joins")
    
    print(f"\n📊 Vista previa:")
    print(df_sucursales[['sucursal_nombre', 'zona', 'lat', 'lon', 'ventas_totales']].head())
    
    print(f"\n🎯 Este notebook usará coordenadas REALES para análisis espacial")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Buffers y Spatial Joins: Análisis Espacial Avanzado

### 🎯 ¿Qué es un Buffer?

Un **buffer** (zona de amortiguamiento) es un área alrededor de una geometría.

**Ejemplo:**
```
        [Radio 1km]
    ┌─────────────┐
    │              │
    │      ●       │  <- Sucursal
    │   (buffer)   │
    └─────────────┘
```

**Creación:**
```python
# Buffer de 1000 metros (1 km)
buffer_1km = punto.buffer(1000)  # Requiere CRS en metros (EPSG:3857)
```

---

### 💼 Casos de Uso de Buffers

1. **Área de influencia comercial:**
   * Radio de 1 km alrededor de sucursal
   * ¿Cuántos clientes potenciales hay?

2. **Cobertura de servicios:**
   * ¿Qué zonas cubre cada sucursal?
   * ¿Hay zonas sin cobertura?

3. **Superposición:**
   * ¿Las sucursales se canibalizan?
   * ¿Dónde hay duplicación de cobertura?

---

### 🔗 Spatial Join (sjoin)

Un **spatial join** une dos GeoDataFrames según su **relación espacial**.

**Tipos de relaciones:**

| Operación | Descripción |
|-----------|-------------|
| **intersects** | Geometrías se tocan o superponen |
| **contains** | Geometría A contiene completamente a B |
| **within** | Geometría A está completamente dentro de B |

---

### 🛠️ Sintaxis de Spatial Join

```python
import geopandas as gpd

# Ejemplo: ¿Qué clientes están dentro del buffer de cada sucursal?
resultado = gpd.sjoin(
    gdf_clientes,      # Left GeoDataFrame
    gdf_buffers,       # Right GeoDataFrame
    how='inner',       # Tipo de join (inner, left)
    predicate='within' # Relación espacial
)
```

**Resultado:** DataFrame con clientes + datos de la sucursal que los contiene.

---

### 📊 Flujo de Análisis de Cobertura

**Paso a paso:**

```python
# 1. Crear GeoDataFrame de sucursales
gdf_sucursales = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs='EPSG:4326'
)

# 2. Transformar a metros para buffers
gdf_metros = gdf_sucursales.to_crs(epsg=3857)

# 3. Crear buffers de 1 km
gdf_metros['buffer_1km'] = gdf_metros.geometry.buffer(1000)

# 4. Cambiar geometría activa a buffers
gdf_buffers = gdf_metros.set_geometry('buffer_1km')

# 5. Spatial join con clientes
cobertura = gpd.sjoin(gdf_clientes, gdf_buffers, predicate='within')
```

---

### 💼 Caso de Uso Empresarial: Optimización de Sucursales

**Pregunta:** ¿Dónde abrir la próxima sucursal?

**Análisis:**
1. Crear buffers de 2 km alrededor de sucursales actuales
2. Identificar zonas SIN cobertura
3. Cruzar con datos de población/ingresos
4. Priorizar zonas de alto potencial sin cobertura

---

### ⚠️ Advertencia: CRS

👉 **Buffers requieren CRS en metros** (no grados)  
👉 **Siempre transformar a EPSG:3857 antes de buffer()**  
👉 **Después puedes volver a EPSG:4326 para visualización**

```python
# ❌ MAL: Buffer en grados (distorsionado)
gdf.buffer(0.01)  # 0.01 grados ≠ distancia fija

# ✅ BIEN: Buffer en metros
gdf.to_crs(epsg=3857).buffer(1000)  # 1000 metros = 1 km
```

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🎯 SPATIAL JOINS Y BUFFERS COMERCIALES")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon
    print(f"Versión de GeoPandas: {gpd.__version__}")
except ImportError:
    print("⚠️  GeoPandas no instalado. Ejecuta: %pip install geopandas")

print("\n🎯 En este notebook aprenderás:")
print("  • .buffer(distance) - Crear áreas de influencia")
print("  • gpd.sjoin() - Spatial joins")
print("  • Análisis de cobertura territorial")
print("  • Detección de superposiciones")

print("\n📖 Métodos clave:")
print("  - gdf.to_crs(epsg=3857)  # Transformar a metros")
print("  - gdf.geometry.buffer(1000)  # Buffer 1 km")
print("  - gpd.sjoin(gdf1, gdf2, predicate='within')")
print("  - gdf.overlay(gdf2, how='intersection')")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🎯 Buffers y spatial joins con datos reales de Los Andes Market

### 📐 Buffer comercial: ¿Área de influencia real?

Con las sucursales de **Los Andes Market** podemos crear buffers de 1 km y 2 km alrededor de cada sucursal:

```python
# Convertir a CRS métrico (metros, no grados)
gdf_metric = gdf.to_crs(epsg=3857)

# Crear buffer de 1000 metros (1 km)
gdf_buffers = gdf_metric.copy()
gdf_buffers['geometry'] = gdf_metric.geometry.buffer(1000)
```

---

### ⚠️ Regla crítica: SIEMPRE buffer en CRS métrico

```python
# ❌ MAL: buffer en grados (0.001 grados ≈ 100m, pero varía con latitud)
gdf.buffer(0.001)

# ✅ BIEN: buffer en metros (exacto)
gdf.to_crs(epsg=3857).buffer(1000)  # 1 km exacto
```

---

### 💼 Análisis de cobertura

Con los datos reales podemos responder:
1. **¿Qué sucursales están a menos de 1 km de distancia?** (buffers superpuestos = canibalización)
2. **¿Hay zonas sin cobertura?** (puntos lejanos de cualquier sucursal)
3. **¿Qué sucursal cubre más área?** (menos superposición = más exclusividad)
4. **¿Dónde abrir una nueva sucursal?** (gap en cobertura = oportunidad)

In [0]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import plotly.express as px
import numpy as np

print("🎯 BUFFERS Y SPATIAL JOINS CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None:
    # Crear GeoDataFrame de sucursales
    df_suc = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates()
    ventas_tot = df_ventas.groupby('sucursal_id')['ventas'].sum().reset_index()
    ventas_tot.columns = ['sucursal_id', 'ventas_totales']
    df_suc = df_suc.merge(ventas_tot, on='sucursal_id')

    geometry = [Point(xy) for xy in zip(df_suc['lon'], df_suc['lat'])]
    gdf = gpd.GeoDataFrame(df_suc, geometry=geometry, crs='EPSG:4326')

    print("\n1️⃣  CREAR BUFFERS DE 1 KM Y 2 KM")
    print("-"*70)

    # Convertir a CRS métrico
    gdf_metric = gdf.to_crs(epsg=3857)

    # Buffers
    gdf_buffer_1km = gdf_metric.copy()
    gdf_buffer_1km['geometry'] = gdf_metric.geometry.buffer(1000)

    gdf_buffer_2km = gdf_metric.copy()
    gdf_buffer_2km['geometry'] = gdf_metric.geometry.buffer(2000)

    print(f"\n   Buffers creados: {len(gdf_buffer_1km)} sucursales × 2 radios")
    print(f"   Radio 1 km: área = {gdf_buffer_1km.geometry.iloc[0].area / 1_000_000:.3f} km²")
    print(f"   Radio 2 km: área = {gdf_buffer_2km.geometry.iloc[0].area / 1_000_000:.3f} km²")
    print(f"   ⚠️  Buffer siempre en CRS métrico (metros), no en grados")

    print("\n" + "="*70)
    print("\n2️⃣  DETECTAR SUPERPOSICIÓN: ¿Hay canibalización?")
    print("-"*70)

    # Verificar superposición entre buffers de 1 km
    superposiciones = []
    for i in range(len(gdf_buffer_1km)):
        for j in range(i+1, len(gdf_buffer_1km)):
            intersec = gdf_buffer_1km.geometry.iloc[i].intersection(gdf_buffer_1km.geometry.iloc[j])
            if not intersec.is_empty:
                area_inter = intersec.area / 1_000_000  # km²
                superposiciones.append({
                    'sucursal_1': gdf_buffer_1km.iloc[i]['sucursal_nombre'],
                    'sucursal_2': gdf_buffer_1km.iloc[j]['sucursal_nombre'],
                    'area_superpuesta_km2': area_inter
                })

    if superposiciones:
        df_super = pd.DataFrame(superposiciones).sort_values('area_superpuesta_km2', ascending=False)
        print(f"\n   ⚠️  {len(df_super)} pares con superposición de buffers (1 km):")
        print(df_super.round(3))
        print("\n   💡 Superposición alta = canibalización potencial")
    else:
        print("\n   ✅ No hay superposición entre buffers de 1 km")
        print("   Las sucursales están suficientemente separadas")

    print("\n" + "="*70)
    print("\n3️⃣  ANÁLISIS DE COBERTURA POR ZONA")
    print("-"*70)

    # Área total cubierta por zona
    cobertura_zona = gdf_buffer_1km.groupby('zona').agg(
        sucursales=('sucursal_id', 'count'),
        area_total_km2=('geometry', lambda x: x.unary_union.area / 1_000_000)
    ).round(3)
    print("\n   Área cubierta por zona (buffer 1 km):")
    print(cobertura_zona)

    print("\n" + "="*70)
    print("\n4️⃣  DISTANCIA MÍNIMA ENTRE SUCURSALES")
    print("-"*70)

    # ¿Cuál es la distancia mínima entre dos sucursales?
    n = len(gdf_metric)
    dist_min = float('inf')
    par_min = None
    for i in range(n):
        for j in range(i+1, n):
            d = gdf_metric.geometry.iloc[i].distance(gdf_metric.geometry.iloc[j]) / 1000
            if d < dist_min:
                dist_min = d
                par_min = (gdf_metric.iloc[i]['sucursal_nombre'], gdf_metric.iloc[j]['sucursal_nombre'])

    print(f"\n   Distancia mínima: {dist_min:.2f} km")
    print(f"   Par más cercano: {par_min[0]} ↔ {par_min[1]}")

    if dist_min < 1.0:
        print(f"   ⚠️  Menos de 1 km de distancia → alta canibalización")
    elif dist_min < 2.0:
        print(f"   🟡 Entre 1 y 2 km → canibalización moderada")
    else:
        print(f"   ✅ Más de 2 km → baja canibalización")

    print("\n" + "="*70)
    print("\n5️⃣  RECOMENDACIÓN: ¿Dónde abrir nueva sucursal?")
    print("-"*70)

    # Encontrar gaps: puntos lejanos de cualquier sucursal
    print("\n   Análisis de gaps de cobertura:")
    print("\n   Para identificar zonas sin cobertura:")
    print("   1. Crear grid de puntos en el área de Mendoza")
    print("   2. Calcular distancia de cada punto a la sucursal más cercana")
    print("   3. Puntos con distancia > 2 km = candidatos para nueva sucursal")
    print(f"\n   Sucursal más aislada (mayor distancia al resto):")
    for i in range(n):
        dists = [gdf_metric.geometry.iloc[i].distance(gdf_metric.geometry.iloc[j]) / 1000
                 for j in range(n) if j != i]
        gdf_metric.iloc[i, gdf_metric.columns.get_loc('dist_min_otra')] = min(dists)

    mas_aislada = gdf_metric.sort_values('dist_min_otra', ascending=False).iloc[0]
    print(f"   {mas_aislada['sucursal_nombre']} ({mas_aislada['zona']})")
    print(f"   Distancia a la sucursal más cercana: {mas_aislada['dist_min_otra']:.2f} km")
    print("\n   💡 Cerca de la sucursal más aislada hay menos competencia → oportunidad")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# 🎯 OPCIONAL: Crear buffers y analizar zonas de influencia

# Descomentar para usar datos reales:
"""
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("💾 Cargando datos georeferenciados desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    # Crear GeoDataFrame de sucursales
    df_sucursales = df_ventas[[
        'sucursal_id', 'sucursal_nombre', 'lat', 'lon', 'zona'
    ]].drop_duplicates()
    
    geometry = [Point(xy) for xy in zip(df_sucursales['lon'], df_sucursales['lat'])]
    gdf = gpd.GeoDataFrame(df_sucursales, geometry=geometry, crs='EPSG:4326')
    
    # Reproyectar a sistema métrico para crear buffers
    gdf_metric = gdf.to_crs('EPSG:22185')  # POSGAR 94 Zona 2 (Mendoza)
    
    # Crear buffers de 1km alrededor de cada sucursal
    gdf_buffers = gdf_metric.copy()
    gdf_buffers['geometry'] = gdf_metric.geometry.buffer(1000)  # 1000 metros = 1km
    
    print(f"✅ Buffers creados:")
    print(f"   • Sucursales: {len(gdf_buffers)}")
    print(f"   • Radio del buffer: 1,000 metros (1 km)")
    
    # Calcular áreas de influencia
    gdf_buffers['area_km2'] = gdf_buffers.geometry.area / 1_000_000  # m2 a km2
    
    print(f"\n📏 Zonas de influencia:")
    for idx, row in gdf_buffers.iterrows():
        print(f"   • {row['sucursal_id']}: {row['area_km2']:.2f} km²")
    
    print(f"\n💡 Variables disponibles:")
    print("   • gdf: GeoDataFrame de sucursales (puntos)")
    print("   • gdf_buffers: GeoDataFrame con buffers de 1km")
    print("   • gdf_metric: GeoDataFrame en sistema métrico (POSGAR 94)")
    
    print(f"\n🔍 Operaciones posibles:")
    print("   1. Verificar overlaps entre buffers (competencia territorial)")
    print("   2. Hacer spatial join con datos censales")
    print("   3. Calcular distancias entre sucursales")
    print("   4. Identificar áreas de cobertura y gaps")
    print("   5. Visualizar en mapa con Folium o Plotly")
    
    # Ejemplo: Detectar overlaps
    print(f"\n🎯 Análisis de Cobertura:")
    for i in range(len(gdf_buffers)):
        for j in range(i+1, len(gdf_buffers)):
            if gdf_buffers.iloc[i].geometry.intersects(gdf_buffers.iloc[j].geometry):
                suc1 = gdf_buffers.iloc[i]['sucursal_id']
                suc2 = gdf_buffers.iloc[j]['sucursal_id']
                overlap_area = gdf_buffers.iloc[i].geometry.intersection(
                    gdf_buffers.iloc[j].geometry
                ).area / 1_000_000
                print(f"   ⚠️  Overlap: {suc1} ↔ {suc2} = {overlap_area:.2f} km²")
    
    display(gdf_buffers[['sucursal_id', 'sucursal_nombre', 'zona', 'area_km2']].head())
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del Notebook 09_03

### ✅ Lo que aprendiste

1. **Creación de Buffers (áreas de influencia):**
   ```python
   # Transformar a metros antes de buffer
   gdf_metric = gdf.to_crs('EPSG:3857')
   gdf_metric['buffer_1km'] = gdf_metric.geometry.buffer(1000)  # 1000 metros = 1 km
   ```

2. **Spatial Joins (sjoin):**
   ```python
   # Unir dos GeoDataFrames por relación espacial
   resultado = gpd.sjoin(gdf_clientes, gdf_buffers,
                         how='inner', predicate='within')
   # predicate: 'within', 'intersects', 'contains'
   ```

3. **Análisis de cobertura territorial:**
   ```python
   # ¿Qué clientes están dentro del área de cada sucursal?
   cobertura = gpd.sjoin(gdf_clientes, gdf_buffers, predicate='within')
   # ¿Qué zonas NO tienen cobertura?
   sin_cobertura = gdf_zonas.overlay(gdf_buffers, how='difference')
   ```

4. **Detección de superposiciones (canibalización):**
   ```python
   # ¿Los buffers de dos sucursales se solapan?
   for i in range(len(gdf_buffers)):
       for j in range(i+1, len(gdf_buffers)):
           if gdf_buffers.iloc[i].geometry.intersects(gdf_buffers.iloc[j].geometry):
               overlap = gdf_buffers.iloc[i].geometry.intersection(
                   gdf_buffers.iloc[j].geometry).area / 1_000_000
               print(f"Overlap: {overlap:.2f} km²")
   ```

5. **Caso integrador:**
   - Buffers de 1 km alrededor de sucursales de Los Andes Market
   - Detección de áreas de canibalización entre sucursales
   - Identificación de zonas sin cobertura para expansión

---

### 🛠️ Guía rápida de Buffers y Spatial Joins

**Caso 1: Buffer de 1 km alrededor de sucursales**
```python
gdf_metric = gdf.to_crs('EPSG:3857')
gdf_buffers = gdf_metric.copy()
gdf_buffers['geometry'] = gdf_metric.geometry.buffer(1000)
```

**Caso 2: Spatial join — clientes dentro del buffer**
```python
cobertura = gpd.sjoin(gdf_clientes, gdf_buffers, predicate='within')
# Cada cliente queda asociado a la sucursal que lo cubre
```

**Caso 3: Detectar superposición entre buffers**
```python
gdf_buffers.iloc[i].geometry.intersects(gdf_buffers.iloc[j].geometry)
```

**Caso 4: Calcular área de superposición**
```python
overlap = gdf_buffers.iloc[i].geometry.intersection(
    gdf_buffers.iloc[j].geometry).area / 1_000_000  # km²
```

**Caso 5: Zonas sin cobertura**
```python
sin_cobertura = gdf_zonas.overlay(gdf_buffers, how='difference')
# Áreas que no están dentro de ningún buffer
```

---

### 🏆 Resumen del Módulo 09

**Aprendiste:**

1. **09_01 - Analítica Geoespacial GeoPandas:** Puntos, geometrías, CRS, mapas básicos
2. **09_02 - GeoDataFrames CRS Shapefiles:** Creación de GeoDataFrames, transformaciones CRS, import/export
3. **09_03 - Spatial Joins y Buffers:** Áreas de influencia, sjoin, cobertura, superposiciones

**Habilidades adquiridas:**
* ✅ Crear buffers comerciales con radio definido
* ✅ Realizar spatial joins entre GeoDataFrames
* ✅ Analizar cobertura territorial y detectar gaps
* ✅ Identificar canibalización entre sucursales
* ✅ Optimizar ubicaciones comerciales con datos espaciales

---


<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🗺️ ¡Módulo 09 Completado!</h3>
  <p><i>"Dominas GeoPandas: desde geometrías básicas hasta spatial joins y buffers comerciales. Ahora puedes tomar decisiones de ubicación basadas en datos espaciales."</i></p>
</div>